# 4.1 Mini Agent Demo

这个 notebook 先不接 LLM，也不做 CLI。目标是用当前 `src/` 里的工具，搭一个最小的 rule-based agent demo。

## 目标

这个 demo 要体现 agent 的最小结构：

1. 接收一个 task。
2. 根据 task 选择一个工具。
3. 执行工具并拿到 observation。
4. 记录 trajectory。
5. 基于 observation 生成 final answer。

这里的工具选择先用简单的 if/else，不使用 LLM。

## 导入依赖

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any, cast


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "prototype" else Path.cwd()
SRC_ROOT = PROJECT_ROOT / "src"

os.environ["SWE_AGENT_JOM_WORKSPACE_ROOT"] = str(PROJECT_ROOT)

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from swe_agent_jom.core.tool_result import ErrorResult, SuccessResult, ToolResult
from swe_agent_jom.memory.trajectory import Trajectory
from swe_agent_jom.tools.file_tool import list_files, read_file
from swe_agent_jom.tools.search_tool import search_code

## 先手动调用工具

先不写 agent，直接调用工具，确认当前工具层能正常返回 `ToolResult`。

In [2]:
list_result = list_files(path="src", pattern="*.py", max_results=5)

print(json.dumps(list_result, ensure_ascii=False, indent=2))

{
  "ok": true,
  "data": {
    "path": "src",
    "pattern": "*.py",
    "count": 5,
    "max_results": 5,
    "truncated": true,
    "entries": [
      {
        "path": "src/swe_agent_jom/__init__.py",
        "type": "file",
        "size_bytes": 29
      },
      {
        "path": "src/swe_agent_jom/config/__init__.py",
        "type": "file",
        "size_bytes": 29
      },
      {
        "path": "src/swe_agent_jom/config/settings.py",
        "type": "file",
        "size_bytes": 1291
      },
      {
        "path": "src/swe_agent_jom/core/__init__.py",
        "type": "file",
        "size_bytes": 37
      },
      {
        "path": "src/swe_agent_jom/core/tool_result.py",
        "type": "file",
        "size_bytes": 848
      }
    ]
  }
}


In [3]:
read_result = read_file("README.md", max_chars=500)

print(json.dumps(read_result, ensure_ascii=False, indent=2))

{
  "ok": true,
  "data": {
    "path": "README.md",
    "size_bytes": 1160,
    "content": "# SWE Agent JOM\n\nA learning-oriented SWE agent prototype. The current codebase focuses on building a safe workspace tool runtime before adding an LLM-driven agent loop.\n\n## Current Status\n\nImplemented:\n\n- `src` package layout\n- unified tool result shape\n- workspace path guards\n- exact allowlisted command execution with `shell=False`\n- model-callable file, search, and command tool wrappers\n- thin trajectory logging\n\nNot implemented yet:\n\n- LLM provider adapters\n- agent executor loop\n- patch appli",
    "truncated": true,
    "max_chars": 500
  }
}


In [4]:
search_result = search_code(
    "resolve_workspace_path",
    path="src",
    pattern="*.py",
    max_results=5,
)

print(json.dumps(search_result, ensure_ascii=False, indent=2))

{
  "ok": true,
  "data": {
    "query": "resolve_workspace_path",
    "path": "src",
    "pattern": "*.py",
    "case_sensitive": false,
    "count": 5,
    "max_results": 5,
    "truncated": true,
    "matches": [
      {
        "path": "src/swe_agent_jom/runtime/command_runner.py",
        "line_number": 13,
        "line": "from swe_agent_jom.runtime.workspace import resolve_workspace_path"
      },
      {
        "path": "src/swe_agent_jom/runtime/command_runner.py",
        "line_number": 49,
        "line": "        resolved_cwd = resolve_workspace_path(cwd)"
      },
      {
        "path": "src/swe_agent_jom/runtime/workspace.py",
        "line_number": 21,
        "line": "def resolve_workspace_path(path: str | Path = \".\") -> Path:"
      },
      {
        "path": "src/swe_agent_jom/runtime/workspace.py",
        "line_number": 52,
        "line": "    resolved = resolve_workspace_path(path)"
      },
      {
        "path": "src/swe_agent_jom/runtime/workspace.py",
    

## 实现一个最小 rule-based agent

这个 agent 只支持一个很窄的任务：搜索代码。

它不是 LLM agent，因为工具选择不是模型决定的；但它已经有 agent loop 的骨架：task -> tool call -> tool result -> final answer。

In [5]:
def summarize_search_result(result: ToolResult) -> str:
    """Create a simple final answer from search_code ToolResult."""
    if not result["ok"]:
        error = cast(ErrorResult, result)
        return f"Search failed: {error['message']}"

    success = cast(SuccessResult, result)
    data = success["data"]
    matches = data["matches"]

    if not matches:
        return f"No matches found for {data['query']!r}."

    first_match = matches[0]

    return (
        f"Found {data['count']} match(es) for {data['query']!r}. "
        f"First match: {first_match['path']}:{first_match['line_number']}"
    )


def run_mini_agent(task: str) -> dict[str, Any]:
    """Run one fixed, rule-based agent turn.

    这个函数故意保持很小：它只演示 agent 如何记录任务、调用工具、记录工具结果，
    再生成最终回答。后续真正接 LLM 时，可以把这里的 if/else 替换成模型决策。
    """
    trajectory = Trajectory()
    trajectory.record_user_message(task)

    query = "resolve_workspace_path"
    tool_arguments = {
        "query": query,
        "path": "src",
        "pattern": "*.py",
        "max_results": 5,
    }

    trajectory.record_assistant_message(
        "I will search the source code for resolve_workspace_path."
    )
    trajectory.record_tool_call("search_code", tool_arguments)

    tool_result = search_code(**tool_arguments)

    trajectory.record_tool_result("search_code", cast(dict[str, Any], tool_result))

    final_answer = summarize_search_result(tool_result)
    trajectory.record_final_answer(final_answer)

    return {
        "task": task,
        "final_answer": final_answer,
        "trajectory": trajectory.to_dicts(),
    }

## 运行 mini agent

In [6]:
demo_result = run_mini_agent(
    "Find where resolve_workspace_path is used in the source code."
)

print(demo_result["final_answer"])

Found 5 match(es) for 'resolve_workspace_path'. First match: src/swe_agent_jom/runtime/command_runner.py:13


## 查看完整轨迹

`trajectory` 是这个 demo 最重要的输出。它展示了 agent 从接收任务到最终回答的完整过程。

In [7]:
print(json.dumps(demo_result["trajectory"], ensure_ascii=False, indent=2))

[
  {
    "kind": "user_message",
    "content": {
      "content": "Find where resolve_workspace_path is used in the source code."
    },
    "created_at": "2026-07-08T03:27:19.960485+00:00"
  },
  {
    "kind": "assistant_message",
    "content": {
      "content": "I will search the source code for resolve_workspace_path."
    },
    "created_at": "2026-07-08T03:27:19.960492+00:00"
  },
  {
    "kind": "tool_call",
    "content": {
      "tool_call_id": null,
      "tool_name": "search_code",
      "arguments": {
        "query": "resolve_workspace_path",
        "path": "src",
        "pattern": "*.py",
        "max_results": 5
      }
    },
    "created_at": "2026-07-08T03:27:19.960494+00:00"
  },
  {
    "kind": "tool_result",
    "content": {
      "tool_call_id": null,
      "tool_name": "search_code",
      "result": {
        "ok": true,
        "data": {
          "query": "resolve_workspace_path",
          "path": "src",
          "pattern": "*.py",
          "case_sensit

## 下一步

这个 notebook 只做了 rule-based mini agent。下一步可以考虑：

1. 支持更多工具，比如 `list_files` 和 `read_file`。
2. 把工具选择逻辑从 if/else 换成 LLM tool calling。
3. 把 agent loop 从 notebook 沉淀到 `src/swe_agent_jom/core/executor.py`。